# 13 — Final Writeup Tables

Pulls from all experiment outputs (E1–E8) to produce publication-ready tables and figures.
One notebook that covers both languages and both models.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["numpy", "pandas", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import ast as ast_mod
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

MODELS = ["QW"]  # add "DS" when available
LANGS = ["P", "R"]
LANG_LABELS = {"P": "Python", "R": "Rust"}
MODEL_LABELS = {"QW": "Qwen", "DS": "DeepSeek"}

EPSILONS = ["0.001", "0.1", "0.5"]
CONSISTENCIES = ["0.2", "0.5", "0.8"]
N_LAYERS = 28

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = Path("/content/drive/MyDrive/DATA/New-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/New-Atlas")

CROSS_DIR = DATA_DIR / "cross"
OUT_DIR = CROSS_DIR / "13_writeup"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_DIR}")
print(f"Output: {OUT_DIR}")

In [ ]:
# Cell 2 – Load all neuron list CSVs

# Classification functions
PYTHON_MODULAR = {"ast__Import", "ast__Break", "ast__Pass",
                  "ast__ImportFrom", "ast__Continue", "ast__Assert"}
RUST_MODULAR = {"rust__Use", "rust__Mod", "rust__Break", "rust__Continue",
                "rust__Return", "rust__Unsafe", "rust__Await"}
RUST_CONSTRUCTS = {
    "rust__For", "rust__While", "rust__Loop", "rust__If", "rust__Match",
    "rust__Fn", "rust__Closure", "rust__Let", "rust__LetMut", "rust__Const",
    "rust__Static", "rust__Struct", "rust__Enum", "rust__TypeAlias",
    "rust__Impl", "rust__Trait", "rust__Use", "rust__Mod",
    "rust__Return", "rust__Break", "rust__Continue",
    "rust__Async", "rust__Await", "rust__Unsafe",
    "rust__Ref", "rust__RefMut", "rust__Deref", "rust__Lifetime",
    "rust__Macro", "rust__Attribute", "rust__QuestionMark",
}

def classify(obj, lang):
    if lang == "P":
        if obj.startswith("builtin__"):
            return "Builtin"
        if obj in PYTHON_MODULAR:
            return "Modular"
        return "Non-modular"
    else:  # R
        if obj in RUST_MODULAR:
            return "Modular"
        if obj in RUST_CONSTRUCTS:
            return "Non-modular"
        return "Object"


frames = {}  # (lang, model, eps, cons) -> DataFrame

for lang in LANGS:
    for model in MODELS:
        prefix = f"{lang}_{model}_"
        for eps in EPSILONS:
            for cons in CONSISTENCIES:
                fname = f"{prefix}4_neuron_list_eps{eps}_cons{cons}_L{ALL_LAYERS_STR}_both.csv"
                path = DATA_DIR / fname
                if not path.exists():
                    # Try split files
                    la = "_".join(str(l) for l in range(14))
                    lb = "_".join(str(l) for l in range(14, 28))
                    pa = DATA_DIR / f"{prefix}4_neuron_list_eps{eps}_cons{cons}_L{la}_both.csv"
                    pb = DATA_DIR / f"{prefix}4_neuron_list_eps{eps}_cons{cons}_L{lb}_both.csv"
                    if pa.exists() and pb.exists():
                        df = pd.concat([pd.read_csv(pa), pd.read_csv(pb)], ignore_index=True)
                    else:
                        continue
                else:
                    df = pd.read_csv(path)

                df["circuit_size"] = df["n_concept_only"] + df["n_both"]
                df["group"] = df["object"].apply(lambda o: classify(o, lang))
                frames[(lang, model, eps, cons)] = df

print(f"Loaded {len(frames)} neuron list files")

In [ ]:
# Cell 3 – Table 1: Concept fraction grid (per language x model)

print("TABLE 1: Concept Fraction across parameter grid")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        print(f"\n--- {label} ---")

        rows = []
        for eps in EPSILONS:
            for cons in CONSISTENCIES:
                df = frames.get((lang, model, eps, cons))
                if df is None:
                    rows.append({"eps": eps, "cons": cons, "Concept CF": "-", "Object CF": "-"})
                    continue

                # Concepts = Modular + Non-modular
                concept_df = df[df["group"].isin(["Modular", "Non-modular"])]
                object_df = df[df["group"].isin(["Builtin", "Object"])]

                c_cf = (concept_df["n_concept_only"].sum() / concept_df["circuit_size"].sum()
                        if concept_df["circuit_size"].sum() > 0 else 0)
                o_cf = (object_df["n_concept_only"].sum() / object_df["circuit_size"].sum()
                        if object_df["circuit_size"].sum() > 0 else 0)

                rows.append({
                    "eps": eps, "cons": cons,
                    "Concept CF": f"{c_cf:.1%}",
                    "Object CF": f"{o_cf:.1%}",
                })

        display(pd.DataFrame(rows))

In [ ]:
# Cell 4 – Table 2: Neuron counts at mid-layer (eps=0.5, cons=0.8)

MID_LAYER = 14
print(f"TABLE 2: Neuron counts at layer {MID_LAYER} (eps=0.5, cons=0.8)")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        df = frames.get((lang, model, "0.5", "0.8"))
        if df is None:
            print(f"\n--- {label}: missing ---")
            continue

        l_mid = df[df["layer"] == MID_LAYER]
        rows = []
        for group_name in sorted(l_mid["group"].unique()):
            g = l_mid[l_mid["group"] == group_name]
            c = g["n_concept_only"].sum()
            s = g["n_both"].sum()
            sz = c + s
            cf = c / sz if sz > 0 else 0
            rows.append({"Group": group_name, "Concept-only": c, "Shared": s,
                         "Circuit size": sz, "Concept fraction": f"{cf:.1%}"})

        print(f"\n--- {label} ---")
        display(pd.DataFrame(rows))

In [ ]:
# Cell 5 – Table 3: Layer profile (eps=0.001, cons=0.8)

print("TABLE 3: Layer-by-layer concept fraction (eps=0.001, cons=0.8)")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        df = frames.get((lang, model, "0.001", "0.8"))
        if df is None:
            print(f"\n--- {label}: missing ---")
            continue

        available_layers = sorted(df["layer"].unique())
        rows = []
        for group_label in sorted(df["group"].unique()):
            row = {"Group": group_label}
            for layer in available_layers:
                sub = df[(df["layer"] == layer) & (df["group"] == group_label)]
                c = sub["n_concept_only"].sum()
                sz = sub["circuit_size"].sum()
                cf = c / sz if sz > 0 else 0
                row[f"L{layer}"] = f"{cf:.1%}"
            rows.append(row)

        print(f"\n--- {label} ---")
        display(pd.DataFrame(rows))

In [ ]:
# Cell 6 – Load and display experiment summaries

print("EXPERIMENT SUMMARIES")
print("=" * 70)

# E3: Meta-circuits
e3_path = CROSS_DIR / "7_E3_meta_circuits" / "7_E3_meta_circuit_results.csv"
if e3_path.exists():
    df_e3 = pd.read_csv(e3_path)
    print(f"\nE3 Meta-circuits: {len(df_e3)} rows")
    sig = df_e3[df_e3["p_value"] < 0.05]
    print(f"  Significant group/layer combos: {len(sig)} / {len(df_e3)}")
else:
    print("\nE3: not available")

# E6: Layer dynamics
e6_path = CROSS_DIR / "7_E6_layer_dynamics" / "7_E6_flow_type_assignments.csv"
if e6_path.exists():
    df_e6 = pd.read_csv(e6_path)
    print(f"\nE6 Flow types:")
    for (lang, model), sub in df_e6.groupby(["lang", "model"]):
        dist = sub["flow_type"].value_counts().to_dict()
        print(f"  {lang}_{model}: {dist}")
else:
    print("\nE6: not available")

# E7: Cross-language
e7_path = CROSS_DIR / "7_E7_cross_language" / "7_E7_cross_language_results.csv"
if e7_path.exists():
    df_e7 = pd.read_csv(e7_path)
    mean_sharing = df_e7.groupby("equivalence_class")["sharing_fraction"].mean()
    print(f"\nE7 Cross-language sharing (mean over layers):")
    for eq, sf in mean_sharing.sort_values(ascending=False).items():
        print(f"  {eq:20s} {sf:.1%}")
else:
    print("\nE7: not available")

# E8: Cross-model
e8_path = CROSS_DIR / "9_E8_cross_model" / "9_E8_cross_model_results.csv"
if e8_path.exists():
    df_e8 = pd.read_csv(e8_path)
    print(f"\nE8 Cross-model:")
    display(df_e8)
else:
    print("\nE8: not available")

In [ ]:
# Cell 7 – Save combined summary

# Combine all tables into one Excel file (if openpyxl available)
try:
    with pd.ExcelWriter(OUT_DIR / "13_all_tables.xlsx") as writer:
        for (lang, model, eps, cons), df in frames.items():
            sheet = f"{lang}_{model}_e{eps}_c{cons}"
            summary = (df.groupby(["object", "group"])
                       .agg(mean_concept_only=("n_concept_only", "mean"),
                            mean_both=("n_both", "mean"),
                            mean_circuit=("circuit_size", "mean"))
                       .reset_index())
            summary["concept_fraction"] = (summary["mean_concept_only"] /
                                           summary["mean_circuit"]).fillna(0)
            summary.to_excel(writer, sheet_name=sheet[:31], index=False)
    print(f"Saved: {OUT_DIR / '13_all_tables.xlsx'}")
except ImportError:
    print("openpyxl not available — saving as CSV instead")
    for (lang, model, eps, cons), df in frames.items():
        fname = f"13_summary_{lang}_{model}_e{eps}_c{cons}.csv"
        summary = (df.groupby(["object", "group"])
                   .agg(mean_concept_only=("n_concept_only", "mean"),
                        mean_both=("n_both", "mean"),
                        mean_circuit=("circuit_size", "mean"))
                   .reset_index())
        summary["concept_fraction"] = (summary["mean_concept_only"] /
                                       summary["mean_circuit"]).fillna(0)
        summary.to_csv(OUT_DIR / fname, index=False)
    print(f"Saved CSVs to {OUT_DIR}")

print("\n13 complete.")